# 09 — Linear Regression Models

Predict player quality changes from team context changes using simple linear regression.

**8 model configs** per position:

| Config | X | Y | OUTCOME? |
|--------|---|---|----------|
| M1 | delta_team (6) | delta_player | No |
| M1b | delta_team (7) | delta_player | Yes |
| M2 | delta_team (6) + pre_player | delta_player | No |
| M2b | delta_team (7) + pre_player | delta_player | Yes |
| M3 | delta_team (6) + pre_player | post_player | No |
| M3b | delta_team (7) + pre_player | post_player | Yes |
| M4 | from_team + to_team (12) + pre_player | post_player | No |
| M4b | from_team + to_team (14) + pre_player | post_player | Yes |

**Train**: to_season 2020-2023 | **Test**: to_season 2024  
**Excluded**: Goalkeepers

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

DATA_ROOT = '/Users/jorgepadilla/Documents/Documents - Jorge\u2019s MacBook Air/thesis_data'
V1 = f'{DATA_ROOT}/processed_data/model_dataset/v_1'

raw = pd.read_parquet(f'{V1}/model_df.parquet')
print(f'Loaded: {raw.shape}')
print(f'Positions: {raw["position"].value_counts().to_string()}')

## 1. Data Prep

In [ ]:
# Drop goalkeepers
df = raw[raw['position'] != 'Goalkeeper'].copy()
n_gk = len(raw) - len(df)
print(f'Dropped {n_gk} GK rows -> {len(df):,} remaining')

# Drop rows with NaN in team qualities (106 from_season=2019 Super Lig)
team_cols = [c for c in df.columns if c.startswith('from_q_proj_') or c.startswith('to_q_') or c.startswith('delta_team_')]
has_team_nan = df[team_cols].isna().any(axis=1)
df = df[~has_team_nan].copy()
print(f'Dropped {has_team_nan.sum()} rows with NaN team qualities -> {len(df):,} remaining')

# All 20 player qualities
ALL_PLAYER_QUALITIES = [
    'Active defence', 'Aerial threat', 'Box threat', 'Chance prevention',
    'Composure', 'Defensive heading', 'Dribbling', 'Effectiveness',
    'Finishing', 'Hold-up play', 'Intelligent defence', 'Involvement',
    'Passing quality', 'Poaching', 'Pressing', 'Progression',
    'Providing teammates', 'Run quality', 'Territorial dominance', 'Winning duels',
]

TEAM_STYLE = ['DEFENCE', 'DEFENSIVE_TRANSITION', 'ATTACKING_TRANSITION',
              'ATTACK', 'PENETRATION', 'CHANCE_CREATION']
TEAM_ALL = TEAM_STYLE + ['OUTCOME']

POSITIONS = sorted(df['position'].unique())
print(f'\nPositions: {POSITIONS}')
print(f'\nTransfers per position:')
print(df['position'].value_counts().to_string())

In [ ]:
# Per position: detect available qualities (null rate < 50%)
position_qualities = {}
for pos in POSITIONS:
    mask = df['position'] == pos
    available = []
    for q in ALL_PLAYER_QUALITIES:
        null_rate = df.loc[mask, f'pre_{q}'].isna().mean()
        if null_rate < 0.50:
            available.append(q)
    position_qualities[pos] = available
    excluded = set(ALL_PLAYER_QUALITIES) - set(available)
    print(f'{pos:20s}: {len(available)} qualities (excluded: {excluded if excluded else "none"})')

In [ ]:
# Train/test split by season
train_mask = df['to_season'].isin([2020, 2021, 2022, 2023])
test_mask = df['to_season'] == 2024

print(f'Train: {train_mask.sum():,} transfers (to_season 2020-2023)')
print(f'Test:  {test_mask.sum():,} transfers (to_season 2024)')
print(f'\nTrain per position:')
print(df[train_mask]['position'].value_counts().to_string())
print(f'\nTest per position:')
print(df[test_mask]['position'].value_counts().to_string())

## 2. Define Model Configs

In [ ]:
def get_model_configs(qualities, team_style=TEAM_STYLE, team_all=TEAM_ALL):
    """Return dict of model_name -> (X_cols, Y_cols) for a given set of player qualities."""
    # Feature column builders
    dt_style = [f'delta_team_{q}' for q in team_style]
    dt_all = [f'delta_team_{q}' for q in team_all]
    pre = [f'pre_{q}' for q in qualities]
    from_proj_style = [f'from_q_proj_{q}' for q in team_style]
    from_proj_all = [f'from_q_proj_{q}' for q in team_all]
    to_curr_style = [f'to_q_{q}' for q in team_style]
    to_curr_all = [f'to_q_{q}' for q in team_all]

    # Target columns
    delta_player = [f'delta_{q}' for q in qualities]
    post_player = [f'post_{q}' for q in qualities]

    return {
        'M1':  (dt_style, delta_player),
        'M1b': (dt_all, delta_player),
        'M2':  (dt_style + pre, delta_player),
        'M2b': (dt_all + pre, delta_player),
        'M3':  (dt_style + pre, post_player),
        'M3b': (dt_all + pre, post_player),
        'M4':  (from_proj_style + to_curr_style + pre, post_player),
        'M4b': (from_proj_all + to_curr_all + pre, post_player),
    }

# Quick check with one position
sample_pos = POSITIONS[0]
sample_configs = get_model_configs(position_qualities[sample_pos])
print(f'Sample configs for {sample_pos}:')
for name, (x_cols, y_cols) in sample_configs.items():
    print(f'  {name:4s}: {len(x_cols)} features -> {len(y_cols)} targets')

## 3. Train & Evaluate All Models

In [ ]:
results = []  # list of dicts with results

for pos in POSITIONS:
    qualities = position_qualities[pos]
    configs = get_model_configs(qualities)
    pos_df = df[df['position'] == pos]

    for model_name, (x_cols, y_cols) in configs.items():
        # Get clean data (drop NaN in features + targets)
        all_cols = x_cols + y_cols
        clean = pos_df[all_cols].dropna()

        # Split
        clean_idx = clean.index
        train_idx = clean_idx[clean_idx.isin(df[train_mask].index)]
        test_idx = clean_idx[clean_idx.isin(df[test_mask].index)]

        if len(train_idx) < 20 or len(test_idx) < 5:
            continue

        X_train = clean.loc[train_idx, x_cols].values
        Y_train = clean.loc[train_idx, y_cols].values
        X_test = clean.loc[test_idx, x_cols].values
        Y_test = clean.loc[test_idx, y_cols].values

        # Fit
        model = LinearRegression()
        model.fit(X_train, Y_train)

        # Predict
        Y_pred = model.predict(X_test)

        # Evaluate per quality
        for i, q in enumerate(qualities):
            y_col = y_cols[i]
            r2 = r2_score(Y_test[:, i], Y_pred[:, i])
            mae = mean_absolute_error(Y_test[:, i], Y_pred[:, i])
            results.append({
                'position': pos,
                'model': model_name,
                'quality': q,
                'r2': r2,
                'mae': mae,
                'n_train': len(train_idx),
                'n_test': len(test_idx),
                'n_features': len(x_cols),
            })

results_df = pd.DataFrame(results)
print(f'Total evaluations: {len(results_df)}')
print(f'Unique models run: {results_df.groupby(["position", "model"]).ngroups}')

## 4. Results: Mean R² by Position x Config

In [ ]:
# Pivot: mean R² per (position, model)
pivot_r2 = results_df.groupby(['position', 'model'])['r2'].mean().unstack('model')
# Reorder columns
col_order = ['M1', 'M1b', 'M2', 'M2b', 'M3', 'M3b', 'M4', 'M4b']
col_order = [c for c in col_order if c in pivot_r2.columns]
pivot_r2 = pivot_r2[col_order]

print('MEAN R² BY POSITION x MODEL CONFIG (test set)')
print(pivot_r2.round(4).to_string())
print(f'\nOverall mean per model:')
print(pivot_r2.mean().round(4).to_string())

In [ ]:
# Heatmap
fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(pivot_r2, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
            linewidths=0.5, ax=ax, vmin=-0.5, vmax=0.5)
ax.set_title('Mean R² by Position x Model Config (test set: to_season 2024)', fontsize=12)
ax.set_ylabel('')
ax.set_xlabel('Model Config')
plt.tight_layout()
plt.show()

In [ ]:
# Also show MAE
pivot_mae = results_df.groupby(['position', 'model'])['mae'].mean().unstack('model')
pivot_mae = pivot_mae[[c for c in col_order if c in pivot_mae.columns]]
print('MEAN MAE BY POSITION x MODEL CONFIG (test set)')
print(pivot_mae.round(4).to_string())
print(f'\nOverall mean MAE per model:')
print(pivot_mae.mean().round(4).to_string())

## 5. R² per Quality (Best Model)

In [ ]:
# Find best model config overall
best_model = pivot_r2.mean().idxmax()
print(f'Best model config (highest mean R²): {best_model}')
print(f'Mean R² = {pivot_r2.mean()[best_model]:.4f}')

# R² per quality for best model, per position
best_results = results_df[results_df['model'] == best_model]
pivot_quality = best_results.pivot_table(index='quality', columns='position', values='r2')
pivot_quality = pivot_quality.reindex(columns=POSITIONS)

print(f'\nR² PER QUALITY — {best_model}')
print(pivot_quality.round(3).to_string())

In [ ]:
# Heatmap of R² per quality for best model
fig, ax = plt.subplots(figsize=(10, 10))
sns.heatmap(pivot_quality, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
            linewidths=0.5, ax=ax, vmin=-0.5, vmax=0.5)
ax.set_title(f'R² per Quality — {best_model} (test set)', fontsize=12)
ax.set_ylabel('Player Quality')
ax.set_xlabel('Position')
plt.tight_layout()
plt.show()

## 6. Coefficient Analysis (Best Model)

In [ ]:
# Re-fit best model per position and extract coefficients
print(f'COEFFICIENT ANALYSIS — {best_model}')
print('=' * 80)

for pos in POSITIONS:
    qualities = position_qualities[pos]
    configs = get_model_configs(qualities)
    x_cols, y_cols = configs[best_model]

    pos_df = df[df['position'] == pos]
    clean = pos_df[x_cols + y_cols].dropna()
    clean_idx = clean.index
    train_idx = clean_idx[clean_idx.isin(df[train_mask].index)]

    X_train = clean.loc[train_idx, x_cols].values
    Y_train = clean.loc[train_idx, y_cols].values

    model = LinearRegression()
    model.fit(X_train, Y_train)

    # Coefficient matrix: (n_targets, n_features)
    coef_df = pd.DataFrame(model.coef_, index=qualities, columns=x_cols)

    # Show only the team-quality features (first N columns)
    team_feature_cols = [c for c in x_cols if 'team' in c or 'q_proj' in c or 'to_q' in c]
    if team_feature_cols:
        print(f'\n{pos} — Team quality coefficients:')
        print(coef_df[team_feature_cols].round(3).to_string())

In [ ]:
# Visualize coefficients for best model, one position
sample_pos = 'Midfielder'
qualities = position_qualities[sample_pos]
configs = get_model_configs(qualities)
x_cols, y_cols = configs[best_model]

pos_df = df[df['position'] == sample_pos]
clean = pos_df[x_cols + y_cols].dropna()
train_idx = clean.index[clean.index.isin(df[train_mask].index)]

model = LinearRegression()
model.fit(clean.loc[train_idx, x_cols].values, clean.loc[train_idx, y_cols].values)

team_feature_cols = [c for c in x_cols if 'team' in c or 'q_proj' in c or 'to_q' in c]
if team_feature_cols:
    coef_team = pd.DataFrame(model.coef_, index=qualities, columns=x_cols)[team_feature_cols]
    # Shorten column names for display
    short_names = {c: c.replace('delta_team_', 'd_').replace('from_q_proj_', 'from_').replace('to_q_', 'to_') for c in team_feature_cols}
    coef_team = coef_team.rename(columns=short_names)

    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(coef_team, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                linewidths=0.5, ax=ax)
    ax.set_title(f'{sample_pos} — {best_model} Coefficients (team features)', fontsize=12)
    ax.set_ylabel('Player Quality (target)')
    ax.set_xlabel('Team Feature')
    plt.tight_layout()
    plt.show()

## 7. Summary

In [ ]:
print('=== FINAL SUMMARY ===')
print(f'\nBest model config: {best_model}')
print(f'Overall mean R²: {pivot_r2.mean()[best_model]:.4f}')
print(f'Overall mean MAE: {pivot_mae.mean()[best_model]:.4f}')
print(f'\nMean R² by position ({best_model}):')
for pos in pivot_r2.index:
    r2 = pivot_r2.loc[pos, best_model]
    print(f'  {pos:20s}: {r2:.4f}')

print(f'\nComparison of all configs (mean R² across positions):')
for model_name in col_order:
    r2 = pivot_r2.mean()[model_name]
    mae = pivot_mae.mean()[model_name]
    print(f'  {model_name:4s}: R²={r2:+.4f}  MAE={mae:.4f}')

print(f'\nKey takeaways:')
print(f'  - Does adding OUTCOME help? M1 vs M1b, M2 vs M2b, etc.')
m1_r2 = pivot_r2.mean()['M1']
m1b_r2 = pivot_r2.mean()['M1b']
print(f'    M1 (no OUTCOME): {m1_r2:.4f} vs M1b (with OUTCOME): {m1b_r2:.4f}')

print(f'  - Does adding pre_player help?')
m2_r2 = pivot_r2.mean()['M2']
print(f'    M1 (delta only): {m1_r2:.4f} vs M2 (+ pre_player): {m2_r2:.4f}')

print(f'  - Deltas vs full features?')
m4_r2 = pivot_r2.mean().get('M4b', pivot_r2.mean().get('M4', float('nan')))
print(f'    M2b (deltas+pre): {pivot_r2.mean()["M2b"]:.4f} vs M4b (full): {m4_r2:.4f}')